# 01 Data setup: WeedyRice-RGBMS-DB → YOLO-seg

Phase 2 ของ Rice-Weed Detection

- ใช้ **CPU runtime** ได้ ไม่ต้องใช้ GPU (เก็บโควตา GPU ไว้เทรน)
- ชุดข้อมูล: WeedyRice-RGBMS-DB (Nguyen et al., 2025), CC BY 4.0 ดู `DATASET.md` ไม่ใช่ข้อมูลของเราเอง
- ข้อมูลทั้งหมดอยู่ใน `data/` ซึ่งไม่เข้า git
- รันได้ซ้ำ: ถ้าดาวน์โหลดแล้วจะข้าม

### ก่อนเปิด notebook (ครั้งแรกครั้งเดียว)
เปิด Terminal แล้วรัน:
```
bash ~/rice-weed-detection/scripts/setup_env.sh
```
จากนั้นใน notebook นี้: **Kernel → Change Kernel → `Python (rwd)`**
(ใช้ env แยก เพื่อไม่ให้ env หลักของ Studio Lab พัง)

## 1. Setup

In [ ]:
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
RAW = ROOT / "data/raw/weedyrice"
OUT = ROOT / "data/weedyrice_seg"
CONFIG = ROOT / "configs/weedyrice.yaml"
PY = sys.executable  # the kernel's python; "!python" would use a different env
print("repo:", ROOT)
print("python:", PY, sys.version.split()[0])
if ".venv" not in PY:
    print("WARNING: kernel is not 'Python (rwd)'. Run scripts/setup_env.sh, then Kernel -> Change Kernel.")

In [ ]:
# ดึงโค้ดล่าสุดจาก GitHub (repo เป็น public ไม่ต้องใช้ token)
!git pull --ff-only

In [ ]:
# ติดตั้ง/อัปเดตแพ็กเกจลง env ของ kernel นี้ (.venv) เท่านั้น
!"{PY}" -m pip install -q -r requirements.txt

In [ ]:
# unit tests ใช้ข้อมูลจำลองเท่านั้น
!"{PY}" -m pytest -q

In [ ]:
!df -h ~
!nvidia-smi 2>/dev/null || echo "no GPU: fine for this notebook"

## 2. Download (ประมาณ 5.3 GB)

- แตกเฉพาะ `RGB`, `Masks`, `Metadata` แล้วลบ zip ทิ้ง
- ใช้เวลานาน ถ้า cell หยุดกลางทาง ให้รันใหม่ได้
- ต้องมีที่ว่างประมาณ 7 GB

In [ ]:
if (RAW / "RGB").exists():
    print("already downloaded:", RAW)
else:
    !"{PY}" -m rwd.download mendeley --dataset-id vt4s83pxx6 --version 1 --out "{RAW}" --include RGB Masks Metadata

## 3. Inspect raw files

ดูชื่อคอลัมน์ใน CSV, จำนวนภาพต่อวัน และชื่อไฟล์ RGB/mask
**ส่งผล cell นี้ให้ Claude** เพื่อใส่ค่า `TODO` ใน `configs/weedyrice.yaml`

In [ ]:
!"{PY}" -m rwd.prepare_weedyrice --config "{CONFIG}" --inspect
for d in ["RGB", "Masks", "Metadata"]:
    files = sorted(p for p in (RAW / d).rglob("*") if p.is_file()) if (RAW / d).exists() else []
    print(f"{d}: {len(files)} files, e.g. {[f.name for f in files[:3]]}")

## 4. Build the YOLO-seg dataset

ต้องแก้ค่า `TODO` ใน `configs/weedyrice.yaml` ก่อน (ถ้ายังไม่แก้ cell นี้จะ error และบอกชื่อคอลัมน์ที่มีจริง)

- แบ่ง train/val/test ตาม **วันถ่าย** ไม่สุ่มตามภาพ
- จบแล้วรัน validator อัตโนมัติ: ต้องขึ้น `OK`
- ตั้ง `OVERWRITE = True` ถ้าจะสร้างใหม่หลังแก้ config

In [ ]:
OVERWRITE = False
flag = "--overwrite" if OVERWRITE else ""
!"{PY}" -m rwd.prepare_weedyrice --config "{CONFIG}" {flag}

In [ ]:
report = json.loads((OUT / "validation_report.json").read_text())
print("OK" if report["ok"] else "FAILED")
for split, r in report["splits"].items():
    print(f"{split:<5} images={r['images']:>5}  with_weed={r['images_with_class']}  instances={r['instances']}  empty={r['empty_labels']}")
print("groups:", report["leakage"]["groups_per_split"])
for w in report["warnings"]:
    print("WARNING:", w)
for e in report["errors"]:
    print("ERROR:", e)

## 5. Look at samples

สีม่วง = polygon ของ weedy rice ที่แปลงจาก mask
เช็คด้วยตาว่า polygon ทับวัชพืชจริง ไม่เลื่อน ไม่กลับด้าน

In [ ]:
import matplotlib.pyplot as plt
from rwd.viz import draw_labels, sample_images

picked = sample_images(OUT / "data.yaml", per_split=3, seed=0)
rows = len(picked)
fig, axes = plt.subplots(rows, 3, figsize=(15, 4 * rows), squeeze=False)
for i, (split, paths) in enumerate(picked.items()):
    for j in range(3):
        ax = axes[i][j]
        ax.axis("off")
        if j < len(paths):
            ax.imshow(draw_labels(paths[j], "segment", num_classes=1))
            ax.set_title(f"{split}: {paths[j].name}", fontsize=9)
plt.tight_layout()
plt.show()

## Next

- ส่งผลข้อ 3 (และข้อ 4 ถ้ารันได้) ให้ Claude
- Phase 3 (เทรน) จะอยู่ใน notebook แยก และต้องใช้ GPU runtime